In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Evidence Pipeline — Crawl, Score, Fuse (RTX 4060 v2)

**Changes from 4090 version:**
- Paths updated to `D:\Pics Can Lie\` (personal machine)
- `jit=False` on CLIP load — fixes MemoryError / bad allocation
- `clip_model.float()` — fixes LayerNorm RuntimeError (do NOT use .half())
- `TIMEOUT = 3` instead of 8 — dead links don't waste 8s each
- **Parallel image downloading** via ThreadPoolExecutor — all links hit simultaneously
- `torch.cuda.empty_cache()` every 500 samples
- Saves every 50 samples instead of 100
- `num_workers=0` on DataLoaders (Windows friendly)
- RAM check printed at startup

**Files needed on D:\\:**
- `links_val.json` ✅ already here
- `deberta_val_scores_v2.csv` ✅ already here
- `clip_finetuned_v2/val_features/` — copy from K:\\
- `models/clip/` — copy from K:\\ (or let it re-download)
- `kaggle_dataset_full/merged_balanced/val.json` — copy from K:\\
- `kaggle_dataset_full/metadata/val.json` — copy from K:\\
- `kaggle_dataset_full/images/` — copy from K:\\

**Expected result:** 88-93% accuracy

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import requests
import time
import clip
import psutil
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import f1_score, accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import pickle, warnings
warnings.filterwarnings('ignore')

# ── RAM check ──
ram = psutil.virtual_memory()
print(f'RAM available : {ram.available / 1024**3:.1f} GB / {ram.total / 1024**3:.1f} GB total')
if ram.available / 1024**3 < 5:
    print('WARNING: Less than 5GB RAM free — close Chrome/other apps before loading CLIP')

# ── Paths ──
PROJECT_ROOT   = str(_cfg.ROOT)
DATASET_ROOT   = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES  = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
CLIP_MODEL_DIR = os.path.join(PROJECT_ROOT, 'models', 'clip')
LINKS_FILE     = os.path.join(PROJECT_ROOT, 'links_val.json')
DEBERTA_FILE   = os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv')
SCORES_CACHE   = os.path.join(PROJECT_ROOT, 'evidence_clip_scores.csv')
FUSION_SAVE    = os.path.join(PROJECT_ROOT, 'fusion_evidence')
VAL_ANN_PATH   = os.path.join(DATASET_ROOT, 'merged_balanced', 'val.json')
VAL_META_PATH  = os.path.join(DATASET_ROOT, 'metadata', 'val.json')

os.makedirs(FUSION_SAVE, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

RAM available : 1.7 GB / 15.7 GB total
Device: cuda
GPU  : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM : 8.0 GB


In [2]:
# ── Load all existing signals ──
print('Loading CLIP features...')
clip_probs = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))
id_df      = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels     = id_df['label'].values

print('Loading DeBERTa scores...')
deb_df       = pd.read_csv(DEBERTA_FILE)
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

print('Loading evidence links...')
with open(LINKS_FILE, 'r') as f:
    links_data = json.load(f)

print('Loading val annotations for ID mapping...')
with open(VAL_ANN_PATH, 'r', encoding='utf-8') as f:
    ann_data = json.load(f)
annotations = ann_data['annotations']

idx_to_id = {str(i): str(a['id']) for i, a in enumerate(annotations)}
id_to_links = {}
for idx, article_id in idx_to_id.items():
    if idx in links_data:
        id_to_links[article_id] = links_data[idx]

with open(VAL_META_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

your_ids = set(id_df['id'].values)
covered  = sum(1 for i in your_ids if i in id_to_links)
print(f'Val samples    : {len(your_ids)}')
print(f'With links     : {covered}/{len(your_ids)} ({covered/len(your_ids)*100:.1f}%)')
print(f'CLIP probs     : {clip_probs.shape}')
print(f'DeBERTa scores : {deb_scores.shape}')
print(f'Labels         : real={int((labels==0).sum())} fake={int((labels==1).sum())}')

Loading CLIP features...
Loading DeBERTa scores...
Loading evidence links...
Loading val annotations for ID mapping...
Val samples    : 3232
With links     : 3209/3232 (99.3%)
CLIP probs     : (5000,)
DeBERTa scores : (5000,)
Labels         : real=2500 fake=2500


In [3]:
# ── Load CLIP (frozen) ──
# jit=False   → fixes MemoryError on load
# .float()    → fixes LayerNorm RuntimeError (do NOT use .half())
print('Loading CLIP ViT-L/14 (jit=False, float32)...')
os.environ['CLIP_DOWNLOAD_ROOT'] = CLIP_MODEL_DIR
clip_model, clip_preprocess = clip.load('ViT-L/14', device=device, jit=False)
clip_model.eval()
clip_model = clip_model.float()
for param in clip_model.parameters():
    param.requires_grad = False

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM after CLIP load: {used:.2f} GB / {total:.1f} GB')
print('CLIP loaded and frozen.')

def get_image_features(image_pil):
    with torch.no_grad():
        img_tensor = clip_preprocess(image_pil).unsqueeze(0).to(device)
        features   = clip_model.encode_image(img_tensor)
        features   = features / features.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return features.cpu().numpy()[0]

def get_text_features(text):
    with torch.no_grad():
        tokens   = clip.tokenize([text], truncate=True).to(device)
        features = clip_model.encode_text(tokens)
        features = features / features.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return features.cpu().numpy()[0]

def cosine_sim(a, b):
    return float(np.dot(a, b))

def resolve_image_path(rel_path):
    return os.path.join(DATASET_ROOT, str(rel_path).replace('visual_news/', 'images/'))

print('Feature extraction functions ready.')

Loading CLIP ViT-L/14 (jit=False, float32)...
VRAM after CLIP load: 1.60 GB / 8.0 GB
CLIP loaded and frozen.
Feature extraction functions ready.


In [4]:
# ── Image downloader — parallel version ──
# Key fix: all links downloaded simultaneously instead of sequentially
# Dead links (27%) no longer block live ones — each times out independently

HEADERS       = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
MAX_EV_IMAGES = 3
TIMEOUT       = 3   # reduced from 8 — dead links only waste 3s each now
MAX_WORKERS   = 6   # parallel download threads

def download_image(url):
    try:
        resp = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        if resp.status_code == 200 and len(resp.content) > 1000:
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            if img.size[0] > 50 and img.size[1] > 50:
                return img
    except:
        pass
    return None

def download_urls_parallel(url_list, max_images):
    """Download up to max_images from url_list — all URLs hit in parallel."""
    results = []
    # Try up to max_images*3 URLs to account for dead links
    candidates = url_list[:max_images * 3]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(download_image, url): url for url in candidates}
        for future in as_completed(futures):
            img = future.result()
            if img is not None:
                results.append(img)
            if len(results) >= max_images:
                break
    return results

def get_evidence_images(links):
    direct_urls = [
        (lp[0] if isinstance(lp, list) else lp)
        for lp in links.get('links_direct_search', [])
    ]
    inv_urls = [
        (lp[0] if isinstance(lp, list) else lp)
        for lp in links.get('links_inv_search', [])
    ]
    direct_imgs = download_urls_parallel(direct_urls, MAX_EV_IMAGES)
    inv_imgs    = download_urls_parallel(inv_urls,    MAX_EV_IMAGES)
    return direct_imgs, inv_imgs

print(f'Downloader ready. Parallel (workers={MAX_WORKERS}), timeout={TIMEOUT}s, max {MAX_EV_IMAGES} per type.')

Downloader ready. Parallel (workers=6), timeout=3s, max 3 per type.


In [5]:
# ── Main evidence scoring loop ──
# Resumes from cache automatically if interrupted

SAVE_EVERY  = 50
CLEAR_EVERY = 500

if os.path.exists(SCORES_CACHE):
    scores_df        = pd.read_csv(SCORES_CACHE)
    scores_df['id']  = scores_df['id'].astype(str)
    already_scored   = set(scores_df['id'].values)
    print(f'Resuming — {len(already_scored)} already scored')
else:
    scores_df      = pd.DataFrame()
    already_scored = set()
    print('Starting fresh')

remaining   = [sid for sid in id_df['id'].values if sid not in already_scored]
new_results = []
no_evidence = 0
errors      = 0

print(f'Samples to score : {len(remaining)}')
print(f'Saves every {SAVE_EVERY} samples — safe to interrupt and resume.')

for i, sample_id in enumerate(tqdm(remaining, desc='Evidence scoring')):
    try:
        if sample_id not in metadata:
            no_evidence += 1
            new_results.append({'id': sample_id, 's2': 0.0, 's3': 0.0, 's4': 0.0, 's5': 0.0, 's6': 0.0})
            continue

        meta     = metadata[sample_id]
        caption  = meta['caption']
        img_path = resolve_image_path(meta['image_path'])
        orig_img = Image.open(img_path).convert('RGB')

        orig_feat    = get_image_features(orig_img)
        caption_feat = get_text_features(caption)

        links = id_to_links.get(sample_id, {})
        direct_imgs, inv_imgs = get_evidence_images(links)

        s2_list, s3_list, direct_feats = [], [], []
        for ev_img in direct_imgs:
            ev_feat = get_image_features(ev_img)
            direct_feats.append(ev_feat)
            s2_list.append(cosine_sim(orig_feat, ev_feat))
            s3_list.append(cosine_sim(caption_feat, ev_feat))

        s4_list, s5_list, inv_feats = [], [], []
        for ev_img in inv_imgs:
            ev_feat = get_image_features(ev_img)
            inv_feats.append(ev_feat)
            s4_list.append(cosine_sim(orig_feat, ev_feat))
            s5_list.append(cosine_sim(caption_feat, ev_feat))

        s6_list = []
        if direct_feats and inv_feats:
            for df in direct_feats:
                for ivf in inv_feats:
                    s6_list.append(cosine_sim(df, ivf))

        if not (direct_imgs or inv_imgs):
            no_evidence += 1

        new_results.append({
            'id': sample_id,
            's2': float(np.mean(s2_list)) if s2_list else 0.0,
            's3': float(np.mean(s3_list)) if s3_list else 0.0,
            's4': float(np.mean(s4_list)) if s4_list else 0.0,
            's5': float(np.mean(s5_list)) if s5_list else 0.0,
            's6': float(np.mean(s6_list)) if s6_list else 0.0,
        })

    except Exception as e:
        errors += 1
        print(f'  ERROR on {sample_id}: {type(e).__name__}: {e}')
        new_results.append({'id': sample_id, 's2': 0.0, 's3': 0.0, 's4': 0.0, 's5': 0.0, 's6': 0.0})

    # Save every 50 samples
    if len(new_results) % SAVE_EVERY == 0:
        partial  = pd.DataFrame(new_results)
        combined = pd.concat([scores_df, partial], ignore_index=True) if not scores_df.empty else partial
        combined.to_csv(SCORES_CACHE, index=False)

    # Clear VRAM cache every 500 samples
    if (i + 1) % CLEAR_EVERY == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()
        used = torch.cuda.memory_allocated() / 1024**3
        print(f'  [VRAM cleared at sample {i+1} — {used:.2f} GB used]')

# Final save
final_scores = pd.DataFrame(new_results)
if not scores_df.empty:
    final_scores = pd.concat([scores_df, final_scores], ignore_index=True)
final_scores.to_csv(SCORES_CACHE, index=False)

print(f'Done. Scored: {len(final_scores)} | No evidence: {no_evidence} | Errors: {errors}')

Resuming — 244 already scored
Samples to score : 4584
Saves every 50 samples — safe to interrupt and resume.


Evidence scoring:  11%|█         | 500/4584 [31:07<3:09:31,  2.78s/it] 

  [VRAM cleared at sample 500 — 1.60 GB used]


Evidence scoring:  22%|██▏       | 1000/4584 [1:01:29<2:41:54,  2.71s/it]

  [VRAM cleared at sample 1000 — 1.60 GB used]


...

  [VRAM cleared at sample 4000 — 1.60 GB used]


Evidence scoring:  98%|█████████▊| 4500/4584 [4:33:55<15:08, 10.81s/it]  

  [VRAM cleared at sample 4500 — 1.60 GB used]


Evidence scoring: 100%|██████████| 4584/4584 [4:45:57<00:00,  3.74s/it]  

Done. Scored: 4834 | No evidence: 41 | Errors: 0


In [6]:
# ── Progress check — run anytime in a separate cell without interrupting Cell 5 ──
import pandas as pd
df  = pd.read_csv(_os.path.join(str(_cfg.ROOT), 'features', 'evidence_clip_scores.csv'))
scored   = len(df)
total    = 5000
non_zero = (df[['s2','s3','s4','s5','s6']].sum(axis=1) != 0).sum()
print(f'Progress     : {scored}/{total} ({scored/total*100:.1f}%)')
print(f'With evidence: {non_zero}/{scored} ({non_zero/max(scored,1)*100:.1f}% got real scores)')
print(f'Zero scores  : {scored - non_zero} (dead links or no evidence)')

Progress     : 4834/5000 (96.7%)
With evidence: 4790/4834 (99.1% got real scores)
Zero scores  : 44 (dead links or no evidence)


In [7]:
# ── Build feature matrix ──
scores_df       = pd.read_csv(SCORES_CACHE)
scores_df['id'] = scores_df['id'].astype(str)
score_lookup    = {row['id']: row for _, row in scores_df.iterrows()}

s2 = np.array([score_lookup.get(i, {}).get('s2', 0.0) for i in id_df['id']])
s3 = np.array([score_lookup.get(i, {}).get('s3', 0.0) for i in id_df['id']])
s4 = np.array([score_lookup.get(i, {}).get('s4', 0.0) for i in id_df['id']])
s5 = np.array([score_lookup.get(i, {}).get('s5', 0.0) for i in id_df['id']])
s6 = np.array([score_lookup.get(i, {}).get('s6', 0.0) for i in id_df['id']])

X = np.stack([clip_probs, clip_sims, deb_scores, s2, s3, s4, s5, s6], axis=1)
print(f'Feature matrix: {X.shape}')
print('Signals: [clip_prob, clip_sim, deberta, s2, s3, s4, s5, s6]')

print()
for name, arr in [('s2 orig vs direct', s2), ('s3 cap vs direct', s3),
                   ('s4 orig vs inv',    s4), ('s5 cap vs inv',    s5),
                   ('s6 cross-evidence', s6)]:
    nz = arr[arr != 0.0]
    if len(nz) > 0:
        print(f'  {name}: mean={nz.mean():.3f} coverage={len(nz)}/{len(arr)}')
    else:
        print(f'  {name}: no data')

print()
print('=' * 55)
print('ABLATION (5-fold logistic regression)')
print('=' * 55)
signal_names = ['clip_prob', 'clip_sim', 'deberta', 's2', 's3', 's4', 's5', 's6']
for i, name in enumerate(signal_names):
    cv = cross_val_score(LogisticRegression(), X[:, i].reshape(-1,1), labels, cv=5, scoring='f1')
    print(f'  {name:<12}: F1={cv.mean():.4f} (+-{cv.std():.4f})')
print()
cv = cross_val_score(LogisticRegression(), X, labels, cv=5, scoring='f1')
print(f'  ALL 8 signals: F1={cv.mean():.4f} (+-{cv.std():.4f})')

Feature matrix: (5000, 8)
Signals: [clip_prob, clip_sim, deberta, s2, s3, s4, s5, s6]

  s2 orig vs direct: mean=0.733 coverage=4925/5000
  s3 cap vs direct: mean=0.236 coverage=4925/5000
  s4 orig vs inv: mean=0.620 coverage=3316/5000
  s5 cap vs inv: mean=0.189 coverage=3316/5000
  s6 cross-evidence: mean=0.574 coverage=3289/5000

ABLATION (5-fold logistic regression)
  clip_prob   : F1=0.8520 (+-0.0067)
  clip_sim    : F1=0.8312 (+-0.0120)
  deberta     : F1=0.4638 (+-0.0274)
  s2          : F1=0.3858 (+-0.1962)
  s3          : F1=0.4303 (+-0.2153)
  s4          : F1=0.5125 (+-0.0722)
  s5          : F1=0.5058 (+-0.0549)
  s6          : F1=0.5114 (+-0.0693)

  ALL 8 signals: F1=0.8539 (+-0.0067)


In [8]:
# ── Train Fusion MLP ──
# num_workers=0 required on Windows

class FusionMLP(nn.Module):
    def __init__(self, input_dim=8, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

X_train, X_val, y_train, y_val = train_test_split(
    X, labels, test_size=0.2, random_state=42, stratify=labels)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

train_ds = TensorDataset(torch.tensor(X_train_sc, dtype=torch.float32),
                          torch.tensor(y_train, dtype=torch.float32))
val_ds   = TensorDataset(torch.tensor(X_val_sc, dtype=torch.float32),
                          torch.tensor(y_val, dtype=torch.float32))
train_ld = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=0)
val_ld   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=0)

model_f   = FusionMLP(input_dim=8).to(device)
optimizer = optim.AdamW(model_f.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_f1, best_epoch, patience_ctr, best_weights = 0.0, 0, 0, None
PATIENCE = 15

print(f'{"Epoch":>6} | {"Loss":>8} | {"Val F1":>8} | {"Val Acc":>8} | Status')
print('-' * 62)

for epoch in range(1, 300):
    model_f.train()
    loss_sum = 0.0
    for xb, yb in train_ld:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_f(xb), yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()

    model_f.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in val_ld:
            p = (torch.sigmoid(model_f(xb.to(device))) > 0.5).long().cpu().numpy()
            preds_all.extend(p)
            labels_all.extend(yb.long().numpy())

    vf1  = f1_score(labels_all, preds_all)
    vacc = accuracy_score(labels_all, preds_all)
    scheduler.step(vf1)

    if vf1 > best_f1:
        best_f1, best_epoch, patience_ctr = vf1, epoch, 0
        best_weights = {k: v.clone() for k, v in model_f.state_dict().items()}
        status = '** BEST **'
    else:
        patience_ctr += 1
        status = f'patience {patience_ctr}/{PATIENCE}'

    if epoch % 10 == 0 or 'BEST' in status:
        print(f'{epoch:>6} | {loss_sum/len(train_ld):>8.4f} | {vf1:>8.4f} | {vacc:>8.4f} | {status}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

model_f.load_state_dict(best_weights)
print(f'Best Val F1: {best_f1:.4f} at epoch {best_epoch}')

 Epoch |     Loss |   Val F1 |  Val Acc | Status
--------------------------------------------------------------
     1 |   0.5216 |   0.8649 |   0.8650 | ** BEST **
     3 |   0.3498 |   0.8657 |   0.8650 | ** BEST **
     4 |   0.3358 |   0.8695 |   0.8670 | ** BEST **
     5 |   0.3320 |   0.8707 |   0.8690 | ** BEST **
     7 |   0.3299 |   0.8745 |   0.8720 | ** BEST **
    10 |   0.3192 |   0.8733 |   0.8720 | patience 3/15
    11 |   0.3198 |   0.8757 |   0.8740 | ** BEST **
    18 |   0.3165 |   0.8770 |   0.8740 | ** BEST **
    20 |   0.3057 |   0.8732 |   0.8710 | patience 2/15
    30 |   0.3102 |   0.8722 |   0.8690 | patience 12/15
Early stopping at epoch 33.
Best Val F1: 0.8770 at epoch 18


In [9]:
# ── Final Evaluation ──
model_f.eval()
all_preds, all_probs, all_labels_v = [], [], []
with torch.no_grad():
    for xb, yb in val_ld:
        logits = model_f(xb.to(device))
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs > 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels_v.extend(yb.long().numpy())

final_f1  = f1_score(all_labels_v, all_preds)
final_acc = accuracy_score(all_labels_v, all_preds)
final_auc = roc_auc_score(all_labels_v, all_probs)

print('=' * 65)
print('FULL SYSTEM FINAL EVALUATION')
print('=' * 65)
print(f'Accuracy : {final_acc*100:.2f}%')
print(f'F1 Score : {final_f1:.4f}')
print(f'AUC-ROC  : {final_auc:.4f}')
print()
print(classification_report(all_labels_v, all_preds, target_names=['REAL', 'FAKE']))
print()
print('=' * 65)
print('COMPARISON WITH LITERATURE')
print('=' * 65)
results = [
    ('NewsCLIPpings baseline',       72.44, 'No evidence'),
    ('VERITE (CLIP ViT-L/14)',        74.40, 'No evidence'),
    ('Your BLIP ITM large',          78.00, 'No evidence'),
    ('COSMOS',                       85.00, 'No evidence'),
    ('Your fine-tuned CLIP alone',   85.60, 'No evidence'),
    ('SNIFFER (InstructBLIP ViT-G)', 88.40, 'Google Entity API'),
    ('MUSE-MLP',                     90.00, 'Google API evidence'),
    ('MUSE AITR',                    93.30, 'Google API evidence'),
    ('YOUR FULL SYSTEM', final_acc*100, 'Abdelnabi et al. links'),
]
print(f'{"System":<45} {"Acc":>6}  Evidence')
print('-' * 75)
for name, acc, ev in results:
    marker = ' <-- YOU' if 'YOUR' in name else ''
    print(f'{name:<45} {acc:>5.2f}%  {ev}{marker}')

print()
if final_acc * 100 > 90.0:
    print('BEATS MUSE-MLP (90.0%)')
if final_acc * 100 > 93.3:
    print('BEATS MUSE AITR (93.3%) -- NEW STATE OF THE ART')

FULL SYSTEM FINAL EVALUATION
Accuracy : 87.40%
F1 Score : 0.8770
AUC-ROC  : 0.9458

              precision    recall  f1-score   support

        REAL       0.89      0.85      0.87       500
        FAKE       0.86      0.90      0.88       500

    accuracy                           0.87      1000
   macro avg       0.87      0.87      0.87      1000
weighted avg       0.87      0.87      0.87      1000


COMPARISON WITH LITERATURE
System                                           Acc  Evidence
---------------------------------------------------------------------------
NewsCLIPpings baseline                        72.44%  No evidence
VERITE (CLIP ViT-L/14)                        74.40%  No evidence
Your BLIP ITM large                           78.00%  No evidence
COSMOS                                        85.00%  No evidence
Your fine-tuned CLIP alone                    85.60%  No evidence
SNIFFER (InstructBLIP ViT-G)                  88.40%  Google Entity API
MUSE-MLP            

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv(_os.path.join(str(_cfg.ROOT), 'features', 'evidence_clip_scores.csv'))
non_zero = (df[['s2','s3','s4','s5','s6']].sum(axis=1) != 0).sum()
print(f'Samples with real evidence : {non_zero}/5000 ({non_zero/50:.1f}%)')
print(f'Samples with zero scores   : {5000 - non_zero}/5000')
print()
print('Score means (non-zero only):')
for col in ['s2','s3','s4','s5','s6']:
    nz = df[col][df[col] != 0]
    print(f'  {col}: mean={nz.mean():.3f}  count={len(nz)}')

Samples with real evidence : 4790/5000 (95.8%)
Samples with zero scores   : 210/5000

Score means (non-zero only):
  s2: mean=0.731  count=4763
  s3: mean=0.235  count=4763
  s4: mean=0.620  count=3205
  s5: mean=0.188  count=3205
  s6: mean=0.573  count=3178


In [12]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 381.6 kB/s eta 0:04:26
   ---------------------------------------- 0.5/101.7 MB 381.6 kB/s eta 0:04:26
   ---------------------------------------- 0.8/101.7 MB 459.9 kB/s eta 0:03:40
   ---------------------------------------- 0.8/101.7 MB 459.9 kB/s eta 0:03:40
   ---------------------------------------- 1.0/101.7 MB 488.8 kB/s eta 0:03:26
   -----------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
# Add derived features that capture the real/fake signal better
import numpy as np
import pandas as pd

# Load your existing arrays (s2-s6 already computed)
# These derived features encode the KEY insight:
# FAKE images: orig image matches evidence (s2 high) BUT caption doesn't (s3 low)
# REAL images: both orig image AND caption match evidence

# 1. Mismatch signal — high s2 but low s3 = suspicious (image matches but caption doesn't)
mismatch_direct = s2 - s3          # high = image matches evidence but caption doesn't
mismatch_inv    = s4 - s5          # same for inverse search

# 2. Consistency — how much do direct and inverse evidence agree
ev_consistency  = s6               # already computed cross-evidence similarity

# 3. Caption-evidence ratio — normalized alignment
eps = 1e-8
cap_img_ratio_direct = s3 / (s2 + eps)   # low = caption poorly explained by evidence
cap_img_ratio_inv    = s5 / (s4 + eps)

# 4. Max evidence signal (instead of mean — best match matters more)
scores_df = pd.read_csv(_os.path.join(str(_cfg.ROOT), 'features', 'evidence_clip_scores.csv'))
scores_df['id'] = scores_df['id'].astype(str)

# Build extended feature matrix
X_ext = np.stack([
    clip_probs,           # s1 — fine-tuned CLIP prob
    clip_sims,            # fine-tuned CLIP similarity
    deb_scores,           # deberta NLI
    s2,                   # orig vs direct evidence
    s3,                   # caption vs direct evidence
    s4,                   # orig vs inverse evidence
    s5,                   # caption vs inverse evidence
    s6,                   # cross-evidence consistency
    mismatch_direct,      # NEW: s2-s3 mismatch
    mismatch_inv,         # NEW: s4-s5 mismatch
    cap_img_ratio_direct, # NEW: caption/image ratio direct
    cap_img_ratio_inv,    # NEW: caption/image ratio inverse
], axis=1)

print(f'Extended feature matrix: {X_ext.shape}')
print('New signals: [mismatch_direct, mismatch_inv, cap_img_ratio_direct, cap_img_ratio_inv]')

# Quick check — are new signals actually discriminative?
print()
print('New signal stats by label:')
for name, arr in [('mismatch_direct', mismatch_direct), 
                   ('mismatch_inv',    mismatch_inv),
                   ('cap_ratio_direct',cap_img_ratio_direct),
                   ('cap_ratio_inv',   cap_img_ratio_inv)]:
    real_mean = arr[labels == 0].mean()
    fake_mean = arr[labels == 1].mean()
    print(f'  {name:<20}: real={real_mean:.3f}  fake={fake_mean:.3f}  diff={fake_mean-real_mean:+.3f}')

Extended feature matrix: (5000, 12)
New signals: [mismatch_direct, mismatch_inv, cap_img_ratio_direct, cap_img_ratio_inv]

New signal stats by label:
  mismatch_direct     : real=0.490  fake=0.489  diff=-0.001
  mismatch_inv        : real=0.286  fake=0.287  diff=+0.001
  cap_ratio_direct    : real=0.319  fake=0.321  diff=+0.002
  cap_ratio_inv       : real=0.204  fake=0.207  diff=+0.003


In [10]:
# ── Save fusion model ──
torch.save(model_f.state_dict(), os.path.join(FUSION_SAVE, 'fusion_weights.pt'))
with open(os.path.join(FUSION_SAVE, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

summary = {
    'final_acc'      : final_acc,
    'final_f1'       : final_f1,
    'final_auc'      : final_auc,
    'best_epoch'     : best_epoch,
    'input_dim'      : 8,
    'signals'        : ['clip_prob', 'clip_sim', 'deberta', 's2', 's3', 's4', 's5', 's6'],
    'beats_muse_mlp' : final_acc > 0.90,
    'beats_muse_aitr': final_acc > 0.933,
    'run_on'         : 'RTX 4060 (personal machine)',
}
with open(os.path.join(FUSION_SAVE, 'results.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Model saved to {FUSION_SAVE}')
print(json.dumps(summary, indent=2))

Model saved to D:\Pics Can Lie\fusion_evidence
{
  "final_acc": 0.874,
  "final_f1": 0.876953125,
  "final_auc": 0.9458120000000001,
  "best_epoch": 18,
  "input_dim": 8,
  "signals": [
    "clip_prob",
    "clip_sim",
    "deberta",
    "s2",
    "s3",
    "s4",
    "s5",
    "s6"
  ],
  "beats_muse_mlp": false,
  "beats_muse_aitr": false,
  "run_on": "RTX 4060 (personal machine)"
}
